# 02 - Limpeza e preparação
Parte da base bruta e gera a base limpa que vai ser usada nas análises.

Passos:
1. checar duplicados
2. montar a data e a hora
3. criar colunas novas (dia da semana, faixa de horário, total de vítimas)
4. remover coordenadas fora de Fortaleza
5. padronizar textos

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))  # pra achar a pasta src

import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import *

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)

In [ ]:
df = gpd.read_parquet(ARQ_BRUTO)
print("linhas antes:", len(df))

## 1. Duplicados

In [ ]:
print("COD_ACIDENTE nulo:", df["COD_ACIDENTE"].isna().sum())
print("COD_ACIDENTE repetido:", df["COD_ACIDENTE"].duplicated().sum())
print("linhas totalmente iguais:", df.drop(columns="geometry").duplicated().sum())

In [ ]:
df = df[~df.drop(columns="geometry").duplicated()].copy()
len(df)

## 2. Data e hora

In [ ]:
df["HORA"].head(10)

In [ ]:
df["DATA"] = pd.to_datetime(dict(year=df["ANO"], month=df["MES"], day=df["DIA"]), errors="coerce")
print("datas inválidas:", df["DATA"].isna().sum())

df["HORA_NUM"] = pd.to_numeric(df["HORA"].astype(str).str.slice(0, 2), errors="coerce")
print("horas inválidas:", df["HORA_NUM"].isna().sum())

In [ ]:
df = df.dropna(subset=["DATA", "HORA_NUM"]).copy()
df["HORA_NUM"] = df["HORA_NUM"].astype(int)
df = df[df["HORA_NUM"].between(0, 23)]

## 3. Colunas novas

In [ ]:
df["DIA_SEMANA"] = df["DATA"].dt.dayofweek.map(dict(enumerate(DIAS_SEMANA)))
df["FIM_DE_SEMANA"] = df["DATA"].dt.dayofweek >= 5

df["PERIODO"] = pd.cut(df["HORA_NUM"], bins=[-1, 5, 11, 17, 23],
                       labels=["Madrugada", "Manhã", "Tarde", "Noite"])

df["TOTAL_VITIMAS"] = df["MORTOS"] + df["FERIDOS"]
df["TOTAL_VEICULOS"] = df[VEICULOS].sum(axis=1)
df["GRAVE"] = (df["MORTOS"] > 0)

## 4. Coordenadas

In [ ]:
fora = ~(df["LATITUDE"].between(LAT_MIN, LAT_MAX) & df["LONGITUDE"].between(LON_MIN, LON_MAX))
print("fora da caixa de Fortaleza:", fora.sum())
df[fora][["LOG1", "LOG2", "LATITUDE", "LONGITUDE"]].head()

In [ ]:
df = df[~fora].copy()
len(df)

## 5. Textos

In [ ]:
colunas_texto = ["LOG1", "LOG2", "TIPOCRUZAMENTO", "CONTROLETRAFEGO", "USOSOLO", "NATUREZA", "SEVERIDADE"]
for c in colunas_texto:
    df[c] = df[c].astype(str).str.strip().str.upper()

for c in ["TIPOCRUZAMENTO", "CONTROLETRAFEGO", "USOSOLO", "NATUREZA", "SEVERIDADE"]:
    print(c, df[c].unique()[:15], "\n")

## Resultado

In [ ]:
print("linhas depois:", len(df))
df.info()

In [ ]:
df.to_parquet(ARQ_LIMPO)
print("salvo em", ARQ_LIMPO)